# NYC Build and Crash — Project overview
**Assembly: Yassin · final baseline: `origin/main` · 2026-11-09.** This notebook assembles the team's merged work and links to its evidence; it does not rerun API downloads or replace contributor notebooks. Pending PRs are explicitly marked. Use VS Code with a project `.venv`, install `pip install -r requirements.txt`, and start the kernel at the repository root. Run the small availability check below, then follow the pipeline order when regenerating data.

## 1. The question and NYC relevance
Do NYC ZIP codes with more building activity also have more reported collisions, and does the relationship vary by construction type, borough, year and crash severity? New Yorkers share streets with deliveries, site access and changing land use; studying where these activities coincide can inform questions for street-safety planning.

We consider three hypotheses: construction activity is associated with greater danger; busy places have both more construction and more traffic; or there is no consistent relationship. The data can distinguish descriptive patterns, but it cannot establish a causal effect. Counts, fatalities per collision and injuries per collision answer different questions; none measures risk per pedestrian, vehicle-mile or resident without an exposure denominator.

## 2. The two datasets and their cleaning evidence
### Housing: DOB NOW: Build Job Application Filings
ID: `w9ak-ipjd` · [NYC Open Data source](https://data.cityofnewyork.us/d/w9ak-ipjd).

This source describes building job **filings**, not completed homes or a census of construction activity. It was selected for filing dates, construction categories, proposed units and location attributes that support aggregation with road-safety data. `job_filing_number` identifies a filing; `job_id` groups a job's filings; the later housing aggregation counts initial `I1` filings only, preventing amendments from being counted as new jobs. Geographic fields include postcode (renamed `zip_code`), borough, coordinates, NTA and census tract. Time fields include `filing_date`, `approved_date`, `first_permit_date` and `current_status_date`; grouping uses the filing year, not the later administrative dates.

### Transportation: Motor Vehicle Collisions — Crashes
ID: `h9gi-nx95` · [NYC Open Data source](https://data.cityofnewyork.us/d/h9gi-nx95).

This source describes police-reported crashes, with `collision_id`, crash date/time, ZIP, borough, coordinates, casualty counts, contributing factors and vehicle types. It was selected to compare construction activity with both collision volume and severity. Reported crashes are not all road incidents; contributing factors are reporting fields, not established causal explanations.

### Before and after: actual saved Phase 1 outputs
“Before” means the scoped table loaded into that stage, rather than the entire live API population. Cell indices below are zero-based JSON cell indices, so a reviewer can locate the exact saved output.

| Stage and evidence notebook | Before: rows × columns | After: rows × columns | Saved output evidence |
|---|---:|---:|---|
| [Housing initial cleaning — Issa](phase1_housing_initial_cleaning_issa.ipynb) | 731,262 × 95 | 730,459 × 25 | cells 5 and 23 |
| [Housing follow-up — Walid](phase1_housing_eda_cleaning_walid.ipynb) | 730,459 × 25 | 730,459 × 34 | cells 1 and 51–52 |
| [Collisions initial cleaning — Yassin](phase1_collisions_initial_cleaning_yassin.ipynb) | 447,156 × 29 | 447,156 × 29 | cells 9 and 20 |
| [Collisions follow-up — Ahmed](phase1_collisions_eda_cleaning_ahmed.ipynb) | 447,156 × 29 | 447,156 × 31 | cells 3 and 50 |

Issa's saved housing run removes 654 records missing required values, 28 exact duplicates and 121 older versions of repeated filing numbers (cells 12, 14 and 16). Optional administrative dates remain missing where legitimate. Walid investigates numeric extremes, chronology, category consistency and location quality, records corrections/flags, and preserves rows. Filings and unique jobs must not be confused.

Yassin's saved initial run retains 125,794 missing-ZIP records (28.13%), has no duplicate removals and preserves the source columns (cells 14 and 18). Ahmed converts 443 placeholder ZIPs to missing, yielding 126,237 missing ZIPs (28.2%), and retains them with a flag; his export adds derived fields while dropping unused source fields (cells 29, 48 and 50). Geographic missingness is a selection limitation, not a zero count. Rare vehicle slots are often legitimately blank; coordinates and casualty counts must not be fabricated.

### Time window and snapshot provenance
The saved initial collision run spans 2022-01-01 through 2026-10-31 (Yassin cell 5); housing filing dates span 2022-01-01 through 2026-10-31 23:00 (Issa cell 5). The agreed analysis window is January 2022 through October 2026. The final year is partial, not a full annual total; approval and source-completeness checks still apply when regenerating upstream notebooks.

**The snapshots differ.** The merged [collision aggregation notebook](phase2_collisions_aggregation_yassin.ipynb), cell 4, explicitly records a later supplied handoff with 448,091 records through 2026-11-03; the date filter retains 447,497 through 2026-10-31. Its cell 6 excludes 126,405 missing-ZIP records (28.247117%) and retains 321,092 ZIP-coded collisions. These later counts, rather than the earlier Phase 1 counts, feed the integrated table. Equal date windows do not freeze a live dataset: records can be added or amended retrospectively. This overview quotes saved outputs and has not refreshed either API.

## 3. Integration and post-join cleaning
**Grain and key.** Both sources are first aggregated to one row per (`zip_code`, `year`). There is no shared row-level identifier between a crash and a building filing, so individual records are never matched on these fields. Housing counts represent jobs started from `I1` filings; collision counts represent crashes with usable ZIPs. See [Walid's housing aggregation](phase2_housing_zip_year_walid.ipynb), [Yassin's collision aggregation](phase2_collisions_aggregation_yassin.ipynb), and [Issa's integration](phase2_data_integration_issa.ipynb).

**Join.** Issa uses an **outer join** with `validate="one_to_one"`, preserving keys observed on either side and checking the union of keys. An inner join would discard unmatched activity; joining raw records would multiply it. `match_status` retains provenance. The saved integration summary (cell 24) shows 1,040 collision ZIP-years and 1,104 housing ZIP-years joining to 1,134 rows before geographic screening.

**Zero-fill rule.** Integer event-count columns on an absent side are filled with zero only because that side's aggregate contains observed events: no aggregate row means no recorded ZIP-coded crash or initial filing in that ZIP-year. It does not imply that no real-world crash occurred. Crashes with missing ZIPs never enter any ZIP-year and cannot be restored with zeros. Noncount fields are not indiscriminately zero-filled. Borough is selected per ZIP from the modal collision label first, with the housing mode as fallback and alphabetical tie-breaking; conflicts are printed. `is_partial_year` is recomputed from year. These are Issa's actual rules in cells 6–7.

**Post-join cleaning.** Issa screens five-digit ZIP strings using NYC prefixes plus named placeholder/non-NYC exceptions and drops unresolved boroughs; this is a documented screen, not a complete USPS validation. The saved summary removes 11 rows, leaving 1,123 rows × 25 columns across 232 ZIPs (cell 24); final columns are printed at export (cell 18). High counts are flagged with per-year IQR fences rather than deleted; `low_data_flag` identifies rows with both fewer than five collisions and fewer than five housing jobs. Derived construction ranks and injuries per collision are added; zero collision denominators remain missing for rates. Types, unique keys, scope, counts, domain rules, CSV reload and website schema are checked. Count totals reconcile with the inputs after the documented ZIP removals (cells 12, 14, 16, 18 and 21). This post-join implementation is credited to Issa, despite the older TEAM_PLAN allocation to Yassin.

## 4. Notebook run order and interfaces
Run the independent housing and collision paths before aggregation, then integration, then the analyses. All paths below are relative to the repository root. Ignored raw/intermediate files must be regenerated or supplied separately; `data/zip_year.csv` is committed. An API refresh can change snapshot counts even within the agreed date window.

| Order | Notebook | Owner | Input | Output | Purpose / status on main |
|---|---|---|---|---|---|
| H1 | [phase1_housing_initial_cleaning_issa.ipynb](phase1_housing_initial_cleaning_issa.ipynb) | Issa | Housing API / `data/housing_raw.csv` cache | `data/housing_step1.csv.gz` | Missing values, duplicates, filing/job distinction |
| H2 | [phase1_housing_eda_cleaning_walid.ipynb](phase1_housing_eda_cleaning_walid.ipynb) | Walid | `data/housing_step1.csv.gz` | `data/housing_step2.csv.gz` | EDA, numeric/category/location/date cleaning and checks |
| C1 | [phase1_collisions_initial_cleaning_yassin.ipynb](phase1_collisions_initial_cleaning_yassin.ipynb) | Yassin | Collision API / validated local cache | `data/collisions_step1.csv.gz` and audit JSON | Initial missingness, duplicates, verified handoff |
| C2 | [phase1_collisions_eda_cleaning_ahmed.ipynb](phase1_collisions_eda_cleaning_ahmed.ipynb) | Ahmed | Collision API / `data/collisions_raw.csv` cache | `data/collisions_step2.csv.gz` | Independent raw pull, EDA, standardization and flags |
| H3 | [phase2_housing_zip_year_walid.ipynb](phase2_housing_zip_year_walid.ipynb) | Walid | `data/housing_step2.csv.gz` | `data/housing_zip_year.csv.gz` | Initial-filing housing metrics per ZIP-year |
| C3 | [phase2_collisions_aggregation_yassin.ipynb](phase2_collisions_aggregation_yassin.ipynb) | Yassin | `data/collisions_step2.csv.gz` | `data/collisions_zip_year.csv` | Scope filter, missing-ZIP exclusions, casualty sums and reconciliation |
| J | [phase2_data_integration_issa.ipynb](phase2_data_integration_issa.ipynb) | Issa | Housing and collision ZIP-year files (`.csv.gz` preferred; `.csv` accepted) | `data/zip_year.csv` | Outer join, post-join cleaning, schema and totals validation |
| R-I | [phase3_analysis_issa.ipynb](phase3_analysis_issa.ipynb) | Issa | `data/zip_year.csv` | `assets/rq/issa_q1.*`, `issa_q2.*` | Growth/change scatter and borough-share heatmap |
| R-W | [phase3_analysis_walid.ipynb](phase3_analysis_walid.ipynb) | Walid | `data/zip_year.csv` | `assets/rq/walid_q1.*`, `walid_q2.*` | Demolition comparison and planned-units scatter |
| R-Y | [phase3_analysis_yassin.ipynb](phase3_analysis_yassin.ipynb) | Yassin | `data/zip_year.csv` | `assets/rq/yassin_q1.*`, `yassin_q2.*` | Fatality comparison and new-building share trend (merged PR #19) |
| R-A | [phase3_analysis_ahmed.ipynb](phase3_analysis_ahmed.ipynb) | Ahmed | `data/zip_year.csv` | `assets/rq/ahmed_q1.*`, `ahmed_q2.*` | Injury-rate quartiles and contributing-factor profiles |

C2 currently retrieves its own raw data; it does not automatically load C1's export. Consequently, the upstream notebooks must not be assumed to share an identical snapshot. The analysis notebooks are independent after J. The availability check is intentionally nonfatal when a contributor's PR has not merged.

In [1]:
from pathlib import Path

# Run at the repository root. Pending PRs print MISSING without blocking the overview.
notebooks = [
    "00_project_overview.ipynb",
    "phase1_housing_initial_cleaning_issa.ipynb",
    "phase1_housing_eda_cleaning_walid.ipynb",
    "phase1_collisions_initial_cleaning_yassin.ipynb",
    "phase1_collisions_eda_cleaning_ahmed.ipynb",
    "phase2_housing_zip_year_walid.ipynb",
    "phase2_collisions_aggregation_yassin.ipynb",
    "phase2_data_integration_issa.ipynb",
    "phase3_analysis_issa.ipynb",
    "phase3_analysis_walid.ipynb",
    "phase3_analysis_yassin.ipynb",
    "phase3_analysis_ahmed.ipynb",
]
for notebook in notebooks:
    print(f"{'OK' if Path(notebook).is_file() else 'MISSING'}  {notebook}")

OK  00_project_overview.ipynb
OK  phase1_housing_initial_cleaning_issa.ipynb
OK  phase1_housing_eda_cleaning_walid.ipynb
OK  phase1_collisions_initial_cleaning_yassin.ipynb
OK  phase1_collisions_eda_cleaning_ahmed.ipynb
OK  phase2_housing_zip_year_walid.ipynb
OK  phase2_collisions_aggregation_yassin.ipynb
OK  phase2_data_integration_issa.ipynb
OK  phase3_analysis_issa.ipynb
OK  phase3_analysis_walid.ipynb
OK  phase3_analysis_yassin.ipynb
OK  phase3_analysis_ahmed.ipynb


## 5. Eight research questions and the available findings
The question wording and chart types below come from contributor notebooks; the quoted findings come verbatim from the merged `assets/rq/*.md` files. They are shortened only at sentence boundaries. Full explanations, uncertainty and limits remain in each notebook and finding file. Full-year comparisons use 2022–2025; partial 2026 appears only where explicitly specified. Missing-ZIP coverage, no exposure denominator, different ZIP sizes/land use, reporting shifts and shared activity levels limit interpretation throughout.


### Issa Q1: Where is construction growing fastest, and do collisions in those ZIP codes move the same way?
**Chart:** Borough-colored change scatter with quadrants · **Notebook:** [phase3_analysis_issa.ipynb](phase3_analysis_issa.ipynb) · **Status:** merged on main.

**Finding:** Across the 209 NYC ZIP codes with enough data in both 2022 and 2025, the change in housing jobs started (I1 filings) and the change in ZIP-coded collisions have a Spearman rank correlation of -0.22 (permutation p = 0.001), a weak negative association: ZIP codes where construction grew tended, if anything, to see collisions fall slightly. The fastest growth was concentrated in Queens (6 of the top 10 ZIP codes). 28 ZIP codes saw both rise, 51 saw both fall, 40 added jobs while crashes fell and 75 lost jobs while crashes rose.

[Full finding](assets/rq/issa_q1.md) · [Chart](assets/rq/issa_q1.png)

### Issa Q2: Does the borough pattern of construction match the borough pattern of collisions over time?
**Chart:** Diverging borough × year heatmap · **Notebook:** [phase3_analysis_issa.ipynb](phase3_analysis_issa.ipynb) · **Status:** merged on main.

**Finding:** From 2022 to 2025, the borough pattern of construction does not match the borough pattern of collisions. Brooklyn has the largest positive gap: its share of the city's ZIP-coded collisions is on average 9.9 percentage points higher than its share of housing jobs started.

[Full finding](assets/rq/issa_q2.md) · [Chart](assets/rq/issa_q2.png)

### Walid Q1: Is demolition activity in a ZIP code associated with more or fewer collisions per housing job?
**Chart:** Box plot with individual ZIP observations · **Notebook:** [phase3_analysis_walid.ipynb](phase3_analysis_walid.ipynb) · **Status:** merged on main.

**Finding:** Across 150 eligible NYC ZIP codes in 2022–2025, the top demolition quartile began at 35.00 demolition jobs and contained 41 ZIPs, compared with 109 in the rest group. The median collision rate was 118.22 versus 101.21 collisions per 100 housing jobs, so the top group was higher by 17.01.

[Full finding](assets/rq/walid_q1.md) · [Chart](assets/rq/walid_q1.png)

### Walid Q2: Do ZIP codes adding many planned dwelling units in new buildings have a different pedestrian injury rate than ZIP codes adding few?
**Chart:** Borough-colored bubble scatter · **Notebook:** [phase3_analysis_walid.ipynb](phase3_analysis_walid.ipynb) · **Status:** merged on main.

**Finding:** Among 214 eligible NYC ZIP codes in 2022–2025, planned dwelling-unit totals ranged from 0 to 4,649, including 47 ZIPs with zero planned units. The Spearman rank correlation between planned units and pedestrians injured per 100 collisions was 0.237, indicating a positive rank association; the two-sided 10,000-permutation p-value was 0.0005, so it was statistically detectable at the 0.05 level.

[Full finding](assets/rq/walid_q2.md) · [Chart](assets/rq/walid_q2.png)

### Yassin Q1: How deadly are the crashes in ZIP codes with a lot of new construction?
**Chart:** Grouped bars with ZIP-bootstrap intervals · **Notebook:** [phase3_analysis_yassin.ipynb](phase3_analysis_yassin.ipynb) · **Status:** merged on main.

**Finding:** Crashes in ZIPs with the most new construction were not clearly deadlier. The top quartile had 2.18 deaths per 1,000 collisions (95% interval 1.93–2.45; 221 deaths / 101,334 collisions) against 2.40 for the bottom quartile (1.30–3.27; 24 / 10,001). For pedestrians and cyclists the rates were 1.22 (1.03–1.42) and 1.50 (0.67–2.30). Both intervals overlap, so the difference could be chance. This is an association only: about 28% of crashes have no ZIP and are excluded, and there is no traffic or population denominator, so these are deaths per reported crash, not a resident's risk.

[Full finding](assets/rq/yassin_q1.md) · [Chart](assets/rq/yassin_q1.png)

### Yassin Q2: Which boroughs account for a larger share of collisions than of construction, and has that changed?
**Chart:** Borough lines; partial final segment dashed · **Notebook:** [phase3_analysis_yassin.ipynb](phase3_analysis_yassin.ipynb) · **Status:** merged on main.

**Finding:** Measured with new buildings, Manhattan has the largest positive gap (collision share minus new-building share), +14.78 percentage points on average over 2022–2025, and Staten Island the largest negative gap (−24.39). Brooklyn stays positive (+9.69 in 2022, +10.51 in 2025), while the Bronx moves from +4.98 to −2.73 and Staten Island narrows from −23.27 to −14.07. Partial 2026 (January–October) looks similar and is not directly comparable to a full year. This is an association only: shares count only crashes with a ZIP (about 28% are excluded), and there is no traffic or population denominator.

[Full finding](assets/rq/yassin_q2.md) · [Chart](assets/rq/yassin_q2.png)

### Ahmed Q1: Are ZIP codes with more new buildings also the ZIP codes where pedestrians and cyclists are hurt more often per collision?
**Chart:** Grouped injury-rate bars with intervals · **Notebook:** [phase3_analysis_ahmed.ipynb](phase3_analysis_ahmed.ipynb) · **Status:** merged on main.

**Finding:** Across the 172 ZIP codes we could analyse (2022-2025, full years only), the 44 ZIPs in the top quartile for new buildings (at least 47 new-building jobs) had 15.5 pedestrian and cyclist injuries per 100 collisions (95% interval 14.3 to 16.7). The 43 ZIPs in the bottom quartile (at most 9.75 jobs) had 21.4 per 100 collisions (95% interval 19.2 to 23.5).

[Full finding](assets/rq/ahmed_q1.md) · [Chart](assets/rq/ahmed_q1.png)

### Ahmed Q2: Do ZIP codes with the most new construction have a different mix of top contributing factors than ZIP codes with the least?
**Chart:** Horizontal stacked profile bars · **Notebook:** [phase3_analysis_ahmed.ipynb](phase3_analysis_ahmed.ipynb) · **Status:** merged on main.

**Finding:** The two profiles are similar. "Driver Inattention/Distraction" is the most common top factor in 89.8% of ZIP-years in the top-quartile ZIPs (176 ZIP-years) and in 94.8% in the bottom-quartile ZIPs (172 ZIP-years); "Failure to Yield Right-of-Way" is second with 9.7% and 2.9%.

[Full finding](assets/rq/ahmed_q2.md) · [Chart](assets/rq/ahmed_q2.png)

**Note on the findings files.** All eight finding files are on main; each `assets/rq/*.md` has the research question on line 1 followed by its finding, which is the format the website reads (fixed in [PR #21](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/21)). Issa's Q2 and Yassin's Q2 are related borough-share questions: Issa compares each borough's share of collisions with its share of all housing jobs started; Yassin uses new buildings only and continues the series into partial 2026.

## 6. Contributions: assignment versus merged evidence
[TEAM_PLAN](docs/TEAM_PLAN.md) records assignments; the merged repository records delivery. The table credits implementations actually on the baseline. Planned deployment, maps or layout responsibilities are not proof of completion.

| Member | Phase 1 merged work | Phase 2 merged work | Phase 3 / website evidence |
|---|---|---|---|
| Mohamed Issa | Housing initial cleaning and portability, [PR #6](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/6) | Integration **and post-join cleaning**, committed integrated table, [PR #12](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/12) | Research notebook/assets, [PR #17](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/17); deployment assigned in TEAM_PLAN, public deployment not verified by this overview |
| Mohamed Walid | Housing EDA/follow-up cleaning, [PR #1](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/1) | Housing aggregation, [PR #11](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/11) | Research notebook/assets, [PR #18](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/18); layout/story assigned in TEAM_PLAN, no separate implementation claim inferred from that assignment |
| Yassin | Collision initial cleaning, [PR #4](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/4) | Collision aggregation [PR #10](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/10), scope/portability follow-up [PR #15](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/15) | Analysis/assets [PR #19](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/19) and Phase 1 prompt-cell [PR #20](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/20) are merged. Overview assembly [PR #22](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/22). Maps were assigned; this overview does not claim a merged map implementation |
| Ahmed Sanad | Collision EDA/follow-up cleaning in `phase1_collisions_eda_cleaning_ahmed.ipynb`, present on main | Website filters/report foundation, subsequently extended in [PR #16](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/16) | Analysis/assets [PR #13](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/13); Dash filters, report, research/story tabs and Vercel packaging [PR #16](https://github.com/Sanad-Manga/NYC-Build-And-Crash/pull/16) |

The older plan assigned Yassin post-integration cleaning; the merged implementation is in Issa's integration notebook, so it is credited to Issa. These ownership statements distinguish source authorship from Ahmed's review and merge coordination. No unmerged contribution is counted as delivered on main.

## 7. AI prompts, deployment and final handoff
**Prompts used:** [AI_PROMPTS.txt](AI_PROMPTS.txt) contains the team's dated entries, attribution, original prompts or explicitly identified summaries, and explanations of how outputs were used. Yassin's overview-assembly prompt is appended there. AI assisted this assembly, source/output cross-checking and the availability check; existing contributor notebooks were not edited.

**Deployment URL:** https://nyc-build-and-crash.vercel.app (also listed in the README)

This overview validates its own source references and the existence of every notebook; it does not certify a fresh online execution of every upstream API notebook.